In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

load_dotenv()
google_api_ey = os.getenv('GOOGLE_API_KEY')
chat = ChatGoogleGenerativeAI(model = 'gemini-2.0-flash', temperature = 0.0)

In [3]:
import pyttsx3
from langchain.agents import tool
@tool
def speak_output(text: str) -> str:
    """Reads out the assistant's response using text-to-speech."""
    try:
        engine = pyttsx3.init()
        engine.setProperty('rate', 150)  # Speed of speech
        voices = engine.getProperty('voices')
        engine.setProperty('voice', voices[1].id)  # Choose male/female voice
        engine.say(text)
        engine.runAndWait()
        return "🗣️ Spoken successfully"
    except Exception as e:
        return f"❌ TTS failed: {e}"


In [4]:
from langchain.agents import initialize_agent, AgentType, tool, load_tools
import datetime
import speech_recognition as sr
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_experimental.utilities import PythonREPL
search = DuckDuckGoSearchRun()
python_repl = PythonREPL
@tool
def get_time(input: str) -> str:
    """Returns Today's Date"""
    return str(datetime.date.today())

@tool
def speech_to_text(input: str) -> str:
    """Converts speech to text"""
    try:
        r = sr.Recognizer()
        with sr.Microphone() as source:
            print("Speak Now...")
            audio = r.listen(source)
        return r.recognize_google(audio)
    except Exception as e:
        return f"Speech Recognition Failed!\n {e}"
    
@tool
def run_python(code: str) -> str:
    """Executes Python code and returns the result."""
    return python_repl.run(input)

In [5]:
tools = load_tools(["llm-math"], llm = chat)
tools += [get_time, speech_to_text, search, run_python]

agent = initialize_agent(
    tools = tools,
    llm = chat,
    agent = AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors = True,
    verbose = True,
)


C:\Users\HARDIK JAIN\AppData\Local\Temp\ipykernel_23056\259459336.py:4: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(


In [8]:
import langchain
langchain.debug = True
if __name__ == '__main__':
    while True:
        query = speech_to_text.invoke("")
        print(f"You said: {query}\n")
        if query:
            if ('exit' or 'bye' or 'quit' or 'goodbye') in (query.lower()):
                break
            try:
               response = agent.invoke(query)
               print(f"Bot: {response}\n")
               speak_output(response['output'])
            except Exception as e:
                print(f"Error: {e}\n")
        else:
            break

[tool/start] [tool:speech_to_text] Entering Tool run with input:
""
Speak Now...
[tool/end] [tool:speech_to_text] [18.07s] Exiting Tool run with output:
"hello hello"
You said: hello hello

[chain/start] [chain:AgentExecutor] Entering Chain run with input:
{
  "input": "hello hello"
}
[chain/start] [chain:AgentExecutor > chain:LLMChain] Entering Chain run with input:
{
  "input": "hello hello",
  "agent_scratchpad": "",
  "stop": [
    "Observation:"
  ]
}
[llm/start] [chain:AgentExecutor > chain:LLMChain > llm:ChatGoogleGenerativeAI] Entering LLM run with input:
{
  "prompts": [
    "System: Answer the following questions as best you can. You have access to the following tools:\n\nCalculator: Useful for when you need to answer questions about math.\nget_time: Returns Today's Date\nspeech_to_text: Converts speech to text\nduckduckgo_search: A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.\nrun_python: